<a href="https://colab.research.google.com/github/Ramprasath-Meganathan/Evaluating-Quantized-LM-Using-LLM-Judge/blob/main/LLM_Judge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluating Quantized Language Models Using an LLM Judge

This notebook evaluates Qwen2.5-1.5B-Instruct in three configurations:

- FP16
- 8-bit LLM.int8()
- 4-bit NF4

The experiments compare model accuracy, GPU memory usage, latency, throughput, formatting failures, and reasoning behavior on the GSM8K dataset.

# Quantized Language Model Evaluation

This notebook runs the experiments for the project
“Evaluating Quantized Language Models Using an LLM Judge.”

It will:

- Clone the GitHub repository.
- Verify the project files.
- Install the required packages.
- Confirm the available GPU.
- Run Qwen2.5-1.5B-Instruct in FP16, 8-bit, and 4-bit NF4.
- Save generated responses and GPU performance metrics.

In [1]:
from pathlib import Path
import os
import sys
import subprocess

REPO_URL = (
    "https://github.com/"
    "Ramprasath-Meganathan/"
    "Evaluating-Quantized-LM-Using-LLM-Judge.git"
)

PROJECT_DIR = Path(
    "/content/Evaluating-Quantized-LM-Using-LLM-Judge"
)

SRC_DIR = PROJECT_DIR / "src"

if not PROJECT_DIR.exists():
    subprocess.run(
        ["git", "clone", REPO_URL, str(PROJECT_DIR)],
        check=True
    )
else:
    print("Repository already exists.")

required_files = [
    "README.md",
    "requirements.txt",
    "src/config.py",
    "src/model_utils.py",
    "src/generation_utils.py",
    "src/run_generation.py",
]

for filename in required_files:
    file_path = PROJECT_DIR / filename

    assert file_path.is_file(), (
        f"Missing project file: {filename}"
    )

os.chdir(PROJECT_DIR)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

(PROJECT_DIR / "data").mkdir(exist_ok=True)
(PROJECT_DIR / "results").mkdir(exist_ok=True)

print("Project directory:", os.getcwd())
print("Repository setup completed.")

Project directory: /content/Evaluating-Quantized-LM-Using-LLM-Judge
Repository setup completed.


## Install Dependencies

The project uses:

- PyTorch for GPU inference.
- Transformers for loading the language model.
- Accelerate for device placement.
- bitsandbytes for 8-bit and 4-bit quantization.
- Datasets for loading and processing GSM8K.
- NumPy and pandas for data processing and analysis.

In [2]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 49.9 MB/s  0:00:00


## Verify GPU Availability

The experiments require an NVIDIA GPU.

The GPU name and available memory are recorded because inference speed and memory usage depend on the hardware used.

In [3]:
!nvidia-smi

Thu Nov 26 06:33:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3,
            2
        ),
        "GB"
    )
else:
    raise RuntimeError("CUDA GPU is not available.")

CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


## Prepare the Dataset

The generation script expects the cleaned GSM8K dataset at:

`data/gsm8k_clean.jsonl`

Each record should contain:

- `problem_id`
- `question`
- `reference_answer`
- `reasoning_step_count`

The same questions must be used for FP16, 8-bit, and 4-bit experiments.

In [5]:
from pathlib import Path
import json

data_path = Path("data/gsm8k_clean.jsonl")

if not data_path.exists():
    from datasets import load_dataset

    print("Dataset not found.")
    print("Downloading GSM8K test split...")

    dataset = load_dataset(
        "openai/gsm8k",
        "main",
        split="test"
    )

    with open(data_path, "w", encoding="utf-8") as file:
        for problem_id, item in enumerate(dataset):
            full_answer = item["answer"]

            if "####" in full_answer:
                reasoning, reference_answer = (
                    full_answer.split("####", 1)
                )

                reference_answer = (
                    reference_answer.strip()
                    .replace(",", "")
                )
            else:
                reasoning = full_answer
                reference_answer = None

            reasoning_step_count = sum(
                1
                for line in reasoning.splitlines()
                if line.strip()
            )

            record = {
                "problem_id": problem_id,
                "question": item["question"],
                "reference_solution": reasoning.strip(),
                "reference_answer": reference_answer,
                "reasoning_step_count": reasoning_step_count,
            }

            file.write(json.dumps(record) + "\n")

    print("Created:", data_path)

else:
    print("Dataset already exists:", data_path)


# Verify the dataset
with open(data_path, "r", encoding="utf-8") as file:
    first_record = json.loads(file.readline())


required_fields = [
    "problem_id",
    "question",
    "reference_answer",
    "reasoning_step_count",
]

for field in required_fields:
    if field not in first_record:
        raise ValueError(
            f"Missing required field: {field}"
        )

print("First dataset record:")
print(first_record)
print("Dataset verification completed.")

Dataset not found.


README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Created: data/gsm8k_clean.jsonl
First dataset record:
{'problem_id': 0, 'question': "Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?", 'reference_solution': 'Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.\nShe makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.', 'reference_answer': '18', 'reasoning_step_count': 2}
Dataset verification completed.


In [ ]:
from pathlib import Path

input_path = Path("data/gsm8k_clean.jsonl")
pilot_path = Path("data/gsm8k_pilot64.jsonl")

with open(input_path, "r", encoding="utf-8") as source:
    pilot_lines = source.readlines()[:64]

with open(pilot_path, "w", encoding="utf-8") as destination:
    destination.writelines(pilot_lines)

print("Pilot dataset created.")

Pilot dataset created.


## Experimental Settings

All model configurations use the same:

- Model family.
- Dataset.
- Problem order.
- Prompt.
- Maximum number of generated tokens.
- Decoding strategy.
- Batch size.
- Random seed.

The only intended experimental difference is the model precision.

In [7]:
from config import (
    MODEL_NAME,
    SEED,
    MAX_NEW_TOKENS,
    BATCH_SIZE,
)

print("Model:", MODEL_NAME)
print("Seed:", SEED)
print("Maximum new tokens:", MAX_NEW_TOKENS)
print("Batch size:", BATCH_SIZE)

Model: Qwen/Qwen2.5-1.5B-Instruct
Seed: 42
Maximum new tokens: 256
Batch size: 1


### FP16 Pilot

In [ ]:
!python src/run_generation.py \
  --precision fp16 \
  --input data/gsm8k_pilot64.jsonl \
  --output results/pilot64_fp16_results.jsonl \
  --metrics results/pilot64_metrics.csv

GPU: Tesla T4
GPU memory: 14.56 GB
Total problems: 10
Already completed: 0
config.json: 100% 660/660 [00:00<00:00, 2.96MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 28.1MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 68.7MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 121MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 148MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

model.safetensors: downloading bytes:   0% 0.00/3.09G [00:00<?, ?B/s]
model.safetensors: downloading bytes:   2% 57.3M/3.09G [00:01<00:37, 80.3MB/s, 4.42MB/s  ]
model.safetensors: downloading bytes:   3% 84.2M/3.09G [00:02<00:40, 73.7MB/s, 6.58MB/s  ]
model.safetensors: downloading bytes:   3% 98.6M/3.09G [00:02<00:33, 88.6MB/s, 7.71MB/s  ]
model.safetensors: downloading bytes:  10% 298M/3.09G [00:03<00:15, 183MB/s, 22.7MB/s  ]
model.safetensors: reconstructing file:   0% 8.55M/3.09G [00:03<20:16, 2.53MB/s,  431kB/s  ]
model.safetensors: downloading bytes:  11% 325M/3.09G [00:0

### 8-bit Pilot

In [ ]:
!python src/run_generation.py \
  --precision int8 \
  --input data/gsm8k_pilot64.jsonl \
  --output results/pilot64_int8_results.jsonl \
  --metrics results/pilot64_metrics.csv

Streaming output truncated to the last 5000 lines.
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: 

### 4-bit NF4 Pilot

In [ ]:
!python src/run_generation.py \
  --precision nf4 \
  --input data/gsm8k_pilot64.jsonl \
  --output results/pilot64_nf4_results.jsonl \
  --metrics results/pilot64_metrics.csv

GPU: Tesla T4
GPU memory: 14.56 GB
Total problems: 10
Already completed: 0
Loading weights: 100% 338/338 [00:01<00:00, 299.40it/s]
Model device: cuda:0
Running warm-up generation...
[1/10] problem_id=0 time=14.77s
[2/10] problem_id=1 time=15.87s
[3/10] problem_id=2 time=15.69s
[4/10] problem_id=3 time=10.08s
[5/10] problem_id=4 time=14.10s
[6/10] problem_id=5 time=15.64s
[7/10] problem_id=6 time=15.82s
[8/10] problem_id=7 time=16.10s
[9/10] problem_id=8 time=15.56s
[10/10] problem_id=9 time=15.72s

Experiment complete.
{
  "model": "Qwen/Qwen2.5-1.5B-Instruct",
  "precision": "nf4",
  "gpu": "Tesla T4",
  "num_dataset_records": 10,
  "completed_this_run": 10,
  "peak_gpu_memory_mb": 1149.3408203125,
  "total_elapsed_seconds": 149.3639814290002,
  "total_generation_seconds": 149.3468497230001,
  "total_output_tokens": 2417,
  "tokens_per_second": 16.183803036240214,
  "max_new_tokens": 256,
  "batch_size": 1,
  "seed": 42
}


###Inspect the pilot

In [12]:
import json
from pathlib import Path

pilot_files = [
    "results/pilot_fp16_results.jsonl",
    "results/pilot_int8_results.jsonl",
    "results/pilot_nf4_results.jsonl",
]

for filename in pilot_files:
    path = Path(filename)

    print("\nFile:", filename)

    if not path.exists():
        print("Missing file.")
        continue

    with open(path, "r", encoding="utf-8") as file:
        result = json.loads(file.readline())

    print("Precision:", result["precision"])
    print("Problem ID:", result["problem_id"])
    print("Response:")
    print(result["response"])


File: results/pilot_fp16_results.jsonl
Precision: fp16
Problem ID: 0
Response:
To solve this problem, let's break it down into steps:

1. **Calculate total eggs laid per day**: Janet's ducks lay 16 eggs per day.

2. **Determine how many eggs Janes eats for breakfast**:
   - She eats 3 eggs for breakfast each morning.
   
3. **Calculate remaining eggs after breakfast**:
   - Remaining eggs = Total eggs - Eggs eaten for breakfast
   - Remaining eggs = 16 - 3 = 13

4. **Subtract the number of eggs used to bake muffins**:
   - She uses 4 eggs to bake muffins.
   
5. **Calculate the final number of eggs available for sale**:
   - Final eggs for sale = Remaining eggs - Eggs used for baking
   - Final eggs for sale = 13 - 4 = 9
   
6. **Calculate earnings from selling the eggs**:
   - Earnings = Number of eggs sold × Price per egg
   - Earnings = 9 × $2 = $18

Therefore, Janet makes $18 every day at the farmers' market.

Final answer: $18

File: results/pilot_int8_results.jsonl
Precision: in

## FP16 Baseline Experiment

The FP16 model serves as the baseline.

Its generated responses and hardware measurements will be used for comparison with the 8-bit and 4-bit configurations.

In [14]:
!python src/run_generation.py \
  --precision fp16 \
  --input data/gsm8k_clean.jsonl \
  --output results/qwen_quantization_l4_fp16.jsonl \
  --metrics results/full_gpu_metrics.csv

Traceback (most recent call last):
  File "/content/Evaluating-Quantized-LM-Using-LLM-Judge/src/run_generation.py", line 9, in <module>
    import torch
  File "/usr/local/lib/python3.13/dist-packages/torch/__init__.py", line 3330, in <module>
    from torch import _meta_registrations
  File "/usr/local/lib/python3.13/dist-packages/torch/_meta_registrations.py", line 9014, in <module>
    _create_binary_float_meta_func(aten.special_chebyshev_polynomial_w)
  File "/usr/local/lib/python3.13/dist-packages/torch/_meta_registrations.py", line 8990, in _create_binary_float_meta_func
    @register_meta(func)
  File "/usr/local/lib/python3.13/dist-packages/torch/_meta_registrations.py", line 73, in wrapper
    pytree.tree_map_(register, op)
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/_pytree.py", line 1617, in tree_map_
    deque(map(func, *flat_args), maxlen=0)  # consume and exhaust the iterable
KeyboardInterrupt
^C


## 8-bit Quantization Experiment

This experiment evaluates the same Qwen2.5-1.5B-Instruct model using 8-bit LLM.int8() quantization.

The dataset, prompt, decoding settings, and problem order remain unchanged.

In [ ]:
!python src/run_generation.py \
  --precision int8 \
  --input data/gsm8k_clean.jsonl \
  --output results/qwen_quantization_l4_int8.jsonl \
  --metrics results/full_gpu_metrics.csv

## 4-bit NF4 Quantization Experiment

This experiment evaluates the same model using 4-bit NF4 quantization.

The results will show whether stronger weight compression affects mathematical accuracy, reasoning behavior, memory usage, and inference speed.

In [10]:
!python src/run_generation.py \
  --precision nf4 \
  --input data/gsm8k_clean.jsonl \
  --output results/qwen_quantization_l4_nf4.jsonl \
  --metrics results/full_gpu_metrics.csv

GPU: Tesla T4
GPU memory: 14.56 GB
Total problems: 1319
Already completed: 0
Loading weights: 100% 338/338 [00:01<00:00, 260.02it/s]
Model device: cuda:0
Running warm-up generation...
Traceback (most recent call last):
  File "/content/Evaluating-Quantized-LM-Using-LLM-Judge/src/run_generation.py", line 218, in <module>
    run(args)
  File "/content/Evaluating-Quantized-LM-Using-LLM-Judge/src/run_generation.py", line 112, in run
    result = generate_response(
  File "/content/Evaluating-Quantized-LM-Using-LLM-Judge/src/generation_utils.py", line 71, in generate_response
    output_ids = model.generate(
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/_contextlib.py", line 124, in decorate_context
    return func(*args, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py", line 3000, in generate
    result = decoding_method(
  File "/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py", line 3228, in _sample
    out

## Verify Generated Results

The experiments should produce:

- `fp16_results.jsonl`
- `int8_results.jsonl`
- `nf4_results.jsonl`
- `gpu_metrics.csv`

The JSONL files contain the generated model responses.

The metrics file contains GPU memory, latency, throughput, and experiment metadata.

In [ ]:
full_files = [
    "results/qwen_quantization_l4_fp16.jsonl",
    "results/qwen_quantization_l4_int8.jsonl",
    "results/qwen_quantization_l4_nf4.jsonl",
]

for result_file in full_files:
    path = Path(result_file)

    print("\nFile:", result_file)

    if not path.exists():
        print("Missing file.")
        continue

    with open(path, "r", encoding="utf-8") as file:
        first_result = json.loads(file.readline())

    print("Precision:", first_result["precision"])
    print("Problem ID:", first_result["problem_id"])
    print("Response:")
    print(first_result["response"])

## Inspect GPU Metrics

The GPU metrics file contains measurements for each precision, including:

- Peak GPU memory.
- Total generation time.
- Number of generated tokens.
- Tokens per second.
- GPU name.

In [ ]:
import pandas as pd

metrics_path = Path("results/gpu_metrics.csv")

if metrics_path.exists():
    metrics_df = pd.read_csv(metrics_path)
    display(metrics_df)
else:
    print("gpu_metrics.csv was not found.")

## Next Evaluation Stage

The generated responses will next be processed to:

- Extract final numerical answers.
- Compare predictions with GSM8K reference answers.
- Calculate exact-match accuracy.
- Measure formatting failures.
- Analyze accuracy by reasoning difficulty.
- Compare FP16, 8-bit, and 4-bit behavior.
- Evaluate the LLM judge.